Setup


In [ ]:
# 1. Install necessary libraries
!pip install -q -U transformers accelerate bitsandbytes tqdm

import torch
import pandas as pd
from tqdm import tqdm
from transformers import pipeline, AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 2. Configuration for T4 GPU (4-bit quantization)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

# 3. Load Model and Tokenizer
# Using Mistral-7B-Instruct as it performs well on German text
model_id = "mistralai/Mistral-7B-Instruct-v0.2"

print("Loading model... this may take a few minutes.")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Create the generator pipeline
generator = pipeline("text-generation", model=model, tokenizer=tokenizer)
print("✅ Generator erfolgreich geladen!")

Loading model... this may take a few minutes.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

✅ Generator erfolgreich geladen!


Model 3: RAG

In [ ]:
# =================================================================
# MODEL 3: RAG (RETRIEVAL AUGMENTED GENERATION) - FULL STACK
# =================================================================

# 1. Install & Import necessary RAG libraries
!pip install -q -U faiss-cpu langchain-community langchain-huggingface sentence-transformers pandas tqdm

import pandas as pd
from tqdm import tqdm
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# --- STEP 1: LOAD THE TEST DATASET ---
print("--- Step 1: Loading Datasets ---")
try:
    df = pd.read_csv('dataset_clean.csv')
    print(f"✅ Loaded {len(df)} questions from dataset_clean.csv")
except FileNotFoundError:
    print("❌ Error: 'dataset_clean.csv' not found. Please upload it to the Colab files.")

# --- STEP 2: PREPARE THE LEGAL KNOWLEDGE BASE (FAISS) ---
print("\n--- Step 2: Creating Vector Index (FAISS) ---")
# Load the local txt file containing the tax laws
loader = TextLoader("tax_laws.txt", encoding='utf-8')
documents = loader.load()

# Split the large text into smaller chunks for the Retriever
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " "]
)
docs = text_splitter.split_documents(documents)

# Using a multilingual model to handle German legal terminology correctly
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(docs, embeddings)
print("✅ FAISS vector index created successfully.")

# --- STEP 3: DEFINE THE INFERENCE FUNCTION ---
def run_rag_inference(input_df, output_filename):
    results = []
    print(f"\nProcessing {len(input_df)} questions for '{output_filename}'...")

    for i, row in tqdm(input_df.iterrows(), total=len(input_df)):
        # A) RETRIEVAL: Search for the 2 most relevant law snippets
        search_results = vectorstore.similarity_search(row['prompt'], k=2)
        context_text = "\n".join([doc.page_content for doc in search_results])
        question_text = row['prompt']

        # B) PROMPT: German instructions for the AI Model (Formatted to avoid SyntaxErrors)
        # We construct the prompt as a clean string variable
        instruction = "Du bist ein Experte für österreichisches Steuerrecht.\nBeantworte die folgende Frage AUSSCHLIESSLICH basierend auf dem bereitgestellten Kontext.\n\n"
        context_block = f"KONTEXT:\n{context_text}\n\n"
        question_block = f"FRAGE:\n{question_text}"

        rag_prompt = f"<s>[INST] {instruction} {context_block} {question_block} [/INST]"

        # C) GENERATION: Use the pre-loaded 'generator' pipeline
        output = generator(
            rag_prompt,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.1,
            pad_token_id=tokenizer.eos_token_id
        )

        # Extract only the newly generated answer
        full_output = output[0]['generated_text']
        answer = full_output.split("[/INST]")[-1].strip()

        results.append({"id": row['id'], "answer": answer})

    # Save results to CSV (submission format)
    pd.DataFrame(results).to_csv(output_filename, index=False)
    print(f"✅ File '{output_filename}' saved successfully.")

# --- STEP 4: EXECUTION ---

# A) TEST RUN (Small sample to verify format and quality)
print("\n--- Starting Test Run (5 Rows) ---")
run_rag_inference(df.head(5), 'test_submission_5.csv')

# B) FULL SUBMISSION RUN (All 644 questions)
# Note: Expect 30-45 minutes runtime on T4 GPU
print("\n--- Starting Full Submission Run (644 Rows) ---")
run_rag_inference(df, 'submission_rag.csv')

print("\n🚀 All tasks complete. Check the file explorer on the left for your CSV files.")

--- Step 1: Loading Datasets ---
✅ Loaded 643 questions from dataset_clean.csv

--- Step 2: Creating Vector Index (FAISS) ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS vector index created successfully.

--- Starting Test Run (5 Rows) ---

Processing 5 questions for 'test_submission_5.csv'...


100%|██████████| 5/5 [00:43<00:00,  8.68s/it]


✅ File 'test_submission_5.csv' saved successfully.

--- Starting Full Submission Run (644 Rows) ---

Processing 643 questions for 'submission_rag.csv'...


100%|██████████| 643/643 [1:29:05<00:00,  8.31s/it]

✅ File 'submission_rag.csv' saved successfully.

🚀 All tasks complete. Check the file explorer on the left for your CSV files.
